In [ ]:
import chipwhisperer as cw

try:
    scope.dis()
    target.dis()
except:
    print()

scope = cw.scope()

target = cw.target(scope, cw.targets.CW305, bsfile=r'cw305_top.bit',force=True)

print("done");


In [2]:
target.vccint_set(1.0)
# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

# run at 10 MHz:
#target.pll.pll_outfreq_set(10E6, 1)
target.pll.pll_outfreq_set(100E6, 1)

target.clkusbautooff = True
target.clksleeptime = 1

target.get_fpga_buildtime()
scope.clock.adc_src = "extclk_x4"
    
import time
for i in range(5):
    
    scope.clock.reset_adc()
    time.sleep(1)
    if scope.clock.adc_locked:
        break 
assert (scope.clock.adc_locked), "ADC failed to lock"

target.get_fpga_buildtime()

'1/19/2026, 14:17'

In [3]:
#######################################################################################
import pyvisa
try:
    rm.close()
    print("rm.close")
except Exception:
    pass

try:
    n9310a.close()
    print("n9310a.close")
except Exception:
    pass

Ampl = 20
rm = pyvisa.ResourceManager()
instruments = rm.list_resources()
# print(f"Connected instruments: {instruments}")
n9310a_address = instruments[-1]
n9310a = rm.open_resource(n9310a_address)
idn = n9310a.query('*IDN?')


print(f'Device ID: {idn}')
n9310a.write(":AMPLITUDE:CW "+str(Ampl)+"dBm")
n9310a.write(":RFOutput:STATE OFF")
#######################################################################################


Device ID: Keysight Technologies,N9310A,CN0116B473,A.02.24


21

In [ ]:
import logging
import logging.handlers
import queue
import sys
import os
import time
import secrets
from datetime import datetime

import numpy as np
from Crypto.Cipher import AES
from tqdm import tqdm

# =========================
# Async Logging Setup 
# =========================
os.makedirs("logs", exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
main_log_file = f"logs/app_{timestamp}.log"
print("file name: ", timestamp)
log_queue = queue.Queue(maxsize=1_000_000)

queue_handler = logging.handlers.QueueHandler(log_queue)

file_handler = logging.FileHandler(
    main_log_file,
    mode="w",
    delay=True
)
file_handler.setLevel(logging.INFO)

console_handler = logging.StreamHandler(sys.stdout)
console_handler.setLevel(logging.WARNING)

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(message)s"
)
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

listener = logging.handlers.QueueListener(
    log_queue,
    file_handler,
    console_handler,
    respect_handler_level=True
)
listener.start()

logger = logging.getLogger("main")
logger.setLevel(logging.INFO)
logger.addHandler(queue_handler)
logger.propagate = False


logger.info("Application started")
logger.info("clk = %i", target.pll.pll_outfreq_get(1))
# =========================
# Helpers
# =========================
def rand_pt_like(n):
    return [secrets.token_bytes(16) for _ in range(n)]


def faulty_byte_indexes(ref: bytes, faulty: bytes):
    return [i for i in range(16) if ref[i] != faulty[i]]


# =========================
# Test Vectors
# =========================
pts = rand_pt_like(200)   #200 plaintexts

key = bytes.fromhex("5e4c8957bf35536b4f9e376e066f78f6")  ##random
logger.info("AES key: %s", key.hex())

target.fpga_write(target.REG_CRYPT_KEY, key[::-1])

# =========================
# Frequency Sweep
# =========================
start_freq = 1e6
end_freq = 10e6
# num_points = 2000
num_points = 5000

frequencies = np.linspace(start_freq, end_freq, num_points)

logger.info(
    "Frequency sweep prepared: %d frequencies selected",
    len(frequencies)
)

# =========================
# Main Experiment Loop
# =========================
experiment_start = time.time()

for pt_idx, pt in enumerate(tqdm(pts, desc="Plaintexts", unit="pt")):

    logger.info(
        "ITER=%d START plaintext=%s",
        pt_idx,
        pt.hex()
    )

    # Load plaintext
    target.fpga_write(target.REG_CRYPT_TEXTIN, pt[::-1])

    # Reference AES
    ct_ref = AES.new(key, AES.MODE_ECB).encrypt(pt)

    n9310a.write(":RFOutput:STATE ON")

    for freq in frequencies: # picked:
#     tqdm(
#         picked,
#         desc=f"Freqs (pt {pt_idx})",
#         unit="Hz",
#         leave=False
#     ):
        n9310a.write(f":FREQ:CW {freq}Hz")
        time.sleep(0.01)

        for trial in range(200):
#         tqdm(
#             range(500),
#             desc="Trials",
#             unit="tr",
#             leave=False
#         ):
            target.usb_trigger_toggle()

            faulty_ct = target.fpga_read(target.REG_CRYPT_CIPHEROUT, 16)[::-1]

            if faulty_ct != ct_ref:
                fault_indexes = faulty_byte_indexes(ct_ref, faulty_ct)
                
                logger.info(
                    "FAULT ITER=%d ct_ref=%s ct_fault=%s freq=%.0f trial=%d bytes=%s",
                    pt_idx,
                    ct_ref.hex(),
                    faulty_ct.hex(),
                    freq,
                    trial,
                    fault_indexes
                )
                    
#                 logger.info(
#                     "FAULT ITER=%d freq=%.0f trial=%d bytes=%s",
#                     pt_idx,
#                     freq,
#                     trial,
#                     fault_indexes
#                 )

    logger.info("ITER=%d END", pt_idx)

# =========================
# Shutdown & Timing
# =========================
total_time = time.time() - experiment_start
logger.info("Application finished in %.2f seconds", total_time)

listener.stop()
n9310a.write(":RFOutput:STATE OFF")